In [ ]:
# =============================================================================
# DACON Aimers 8기 모델 경량화 해커톤 - ZIP중력팀 최종 제출 코드
# =============================================================================
#
# [개발 환경]
# - OS: Ubuntu 22.04 (Google Colab)
# - GPU: NVIDIA L4 (평가 서버) / NVIDIA T4 (개발 환경)
# - Python: 3.12
# - CUDA: 12.x
#
# [라이브러리 버전]
# [개발 환경 - 코드 작성]
# - torch: 2.6.0+cu124 (개발 환경)
# - transformers: 4.57.3
# - llmcompressor: 0.9.0.2 (자동 설치)
# - compressed-tensors: 0.13.0 (llmcompressor 의존성)
# - datasets: 최신 버전 (자동 설치)
# - accelerate: 최신 버전 (자동 설치)
# - safetensors: 0.7.0
#
# [평가 서버 환경]
# - torch: 2.9.0+cu128
# - transformers: 4.57.3
# - vllm: 0.14.1
# - compressed-tensors: 0.13.0
#
# [외부 데이터]
# 1. KoAlpaca-v1.1a
#    - 출처: https://huggingface.co/datasets/beomi/KoAlpaca-v1.1a
#    - 라이센스: Apache 2.0
#    - 용도: Calibration 데이터 (한국어 60%)
#
# 2. MMLU (Massive Multitask Language Understanding)
#    - 출처: https://huggingface.co/datasets/cais/mmlu
#    - 라이센스: MIT
#    - 용도: Calibration 데이터 (영어 40%)
#
# [기본 모델]
# - LGAI-EXAONE/EXAONE-4.0-1.2B
#   출처: https://huggingface.co/LGAI-EXAONE/EXAONE-4.0-1.2B
#   라이센스: EXAONE AI Model License Agreement 1.1
#
# [경량화 전략]
# - 기법: PTQ (Post-Training Quantization) - W8A8 INT8
# - 도구: llmcompressor QuantizationModifier (DataFree Pipeline)
# - 보호 레이어: lm_head (총 1개)
# - 결과: 2.56GB → 1.29GB (약 50% 압축), 최종 점수 0.6343
#
# [재현 방법]
# - 로컬/서버: 결과물은 ./submit_ZIP중력.zip 생성
# =============================================================================

import subprocess

# torchvision 충돌 방지 (Colab 환경)
subprocess.run(["pip", "uninstall", "-y", "torchvision"], check=True)

# 필수 라이브러리 설치
subprocess.run(
    ["pip", "install", "-q", "llmcompressor==0.9.0.2", "datasets", "--upgrade"],
    check=True
)

import os
import gc
import json
import shutil
import torch
from datasets import load_dataset, Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import QuantizationModifier

gc.collect()
torch.cuda.empty_cache()

In [ ]:
# =============================================================================
# 설정
# =============================================================================
MODEL_ID   = "LGAI-EXAONE/EXAONE-4.0-1.2B"
SAVE_DIR   = "./model"
ZIP_NAME   = "submit_ZIP중력"

NUM_SAMPLES = 512   # Calibration 샘플 수 (DataFree라 실제 양자화에 미사용)
MAX_SEQ_LEN = 1024  # 최대 시퀀스 길이

In [ ]:
# 모델 및 토크나이저 로드
print("모델 로드")

try:
    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_ID,
        trust_remote_code=True,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True,
        low_cpu_mem_usage=True,
    )
    num_layers = model.config.num_hidden_layers
    print(f"  ✓ hidden layers: {num_layers}")
except Exception as e:
    raise RuntimeError(f"[FAIL] 모델 로드 실패: {e}")

In [ ]:
# Calibration 데이터 준비
# - DataFree Pipeline으로 동작하여 실제 양자화에는 사용되지 않음
# - weight의 최대/최소값 통계 참조용
# - 한국어(KoAlpaca) 60% + 영어(MMLU) 40% 혼합
print(f"Calibration 데이터 준비 ({NUM_SAMPLES}개)")

try:
    samples = []
    num_kr = int(NUM_SAMPLES * 0.6)

    # 한국어 데이터: KoAlpaca-v1.1a (Apache 2.0)
    ds_kr = load_dataset("beomi/KoAlpaca-v1.1a", split="train", streaming=True)
    for ex in ds_kr:
        if len(samples) >= num_kr:
            break
        if "instruction" not in ex:
            continue
        msgs = [{"role": "user", "content": ex["instruction"]}]
        formatted = tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=True
        )
        if 50 < len(formatted) < 3000:
            samples.append({"text": formatted})

    # 영어 데이터: MMLU (MIT)
    ds_mmlu = load_dataset("cais/mmlu", "all", split="test", streaming=True)
    for ex in ds_mmlu:
        if len(samples) >= NUM_SAMPLES:
            break
        q = ex["question"]
        choices = ex["choices"]
        choice_text = "\n".join([f"{chr(65+i)}. {c}" for i, c in enumerate(choices)])
        text = f"Question: {q}\n{choice_text}\nAnswer:"
        msgs = [{"role": "user", "content": text}]
        formatted = tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=True
        )
        if 50 < len(formatted) < 3000:
            samples.append({"text": formatted})

    final_ds = Dataset.from_list(samples[:NUM_SAMPLES])
    print(f"  Calibration samples: {len(final_ds)}개")
except Exception as e:
    raise RuntimeError(f"[FAIL] 데이터 준비 실패: {e}")

gc.collect()

In [ ]:
# W8A8 INT8 양자화 적용
#
# [보호 레이어]
# - lm_head: 최종 토큰 예측 담당, 양자화 시 출력 품질 직결
#
# [W8A8]
# - W8: weight 8bit 압축 → 모델 크기 절반 (2.56GB → 1.29GB)
# - A8: activation 8bit → L4 GPU INT8 하드웨어 가속 → 추론 속도 향상
print("W8A8 양자화 적용")

try:
    ignore_list = ["lm_head"]

    print(f"  FP16 보호 ({len(ignore_list)}개): {ignore_list}")
    print(f"  W8A8 적용: layers.0~28 ({num_layers - len(ignore_list)}개)")

    recipe = QuantizationModifier(
        targets="Linear",
        scheme="W8A8",
        ignore=ignore_list,
    )

    oneshot(
        model=model,
        dataset=final_ds,
        recipe=recipe,
        max_seq_length=MAX_SEQ_LEN,
        num_calibration_samples=len(final_ds),
    )
    print("양자화 완료")
except Exception as e:
    raise RuntimeError(f"[FAIL] 양자화 실패: {e}")

gc.collect()
torch.cuda.empty_cache()

In [ ]:
# 모델 저장 및 ZIP 생성
print(f"저장 → {SAVE_DIR}/")

try:
    if os.path.exists(SAVE_DIR):
        shutil.rmtree(SAVE_DIR)
    os.makedirs(SAVE_DIR, exist_ok=True)

    # 압축된 형태로 저장 (safetensors)
    model.save_pretrained(SAVE_DIR, save_compressed=True)
    tokenizer.save_pretrained(SAVE_DIR)

    # generation_config 저장 (평가 서버 호환)
    with open(os.path.join(SAVE_DIR, "generation_config.json"), "w") as f:
        json.dump({"do_sample": False, "transformers_version": "4.57.3"}, f)

    # ZIP 생성
    if os.path.exists(f"{ZIP_NAME}.zip"):
        os.remove(f"{ZIP_NAME}.zip")
    shutil.make_archive(ZIP_NAME, "zip", root_dir=".", base_dir="model")

    zip_size_gb = os.path.getsize(f"{ZIP_NAME}.zip") / 1024 ** 3
    total_bytes = sum(
        os.path.getsize(os.path.join(SAVE_DIR, f))
        for f in os.listdir(SAVE_DIR)
    )
    total_gb = total_bytes / 1024 ** 3

except Exception as e:
    raise RuntimeError(f"[FAIL] 저장 실패: {e}")

print("=" * 60)
print("  최종 결과")
print("=" * 60)
print(f"  모델 크기  : {total_gb:.3f} GB (원본 2.56GB → {total_gb:.3f}GB)")
print(f"  ZIP 크기   : {zip_size_gb:.3f} GB")
print(f"  보호 레이어: lm_head ({len(ignore_list)}개)")
print(f"  양자화 방식: W8A8 DataFree PTQ")
print(f"  최종 점수  : 0.6343")
print("=" * 60)
print(f"{ZIP_NAME}.zip 생성 완료")